# PI — 예측 분위수와 통합성적표 만들기

시계열 CSV 한 개를 넣으면 모델을 고르고, 첫 origin부터 모든 origin의 **예측 분위수(딕셔너리 1)** 와 **통합성적표(딕셔너리 2)** 를 만들어 줍니다. 이 폴더 하나만 있으면 실행됩니다(모델 선정 엔진을 복사해서 같이 넣어 두었습니다).


## 실행

PI_module 폴더에서 실행합니다.

```
python PI.py <CSV 이름> <첫 origin의 마지막 분기 날짜>
python PI.py gold_yoy '2024-01-01'
```

- 날짜의 작은따옴표는 붙여도 되고 빼도 됩니다.
- CSV는 이름만 주면 현재 폴더, `PI_module/data/` 순으로 찾습니다(`.csv`는 생략 가능). 경로를 직접 줘도 됩니다.
- 결과는 화면에 출력되고 `PI_module/output/<이름>/`에 JSON(딕셔너리 그대로)과 CSV(표)로 저장됩니다.

```
선정 모델: GARCH (d=0, D=0) 원 스케일, 리샘플링 구간 | 후보 6/24개, 보정 origin 15개
딕셔너리 1: origin 25개 (2019-Q4 ~ 2025-Q4), 값은 9 x 4 배열 ... 첫 origin 2019-Q4:
{'2019-Q4': [[-151.52, -192.44, ...], ...]}
딕셔너리 2: {'mse': 1.5448, 'crps': 0.7453, 'coverage': 82.1429}
저장(PI_module/output/X1_l1/): X1_l1_forecast.json, ..._score.json, ..._forecast.csv, ..._score.csv
```

화면에는 딕셔너리 2 전체와 딕셔너리 1의 첫 origin만 나오고, 딕셔너리 1 전체는 `*_forecast.json`에 있습니다.


## 입력

**CSV**: `Date`, `Data` 두 열, 분기 자료, 연속된 분기(빠지거나 겹치는 분기가 없어야 함).

```
Date,Data
2011-06-01,70.10
2011-09-01,-220.88
...
```

- `Date`는 **그 날짜가 속한 분기**로 읽습니다. 2018-12-01은 2018-Q4, 2024-01-01은 2024-Q1입니다.
- 값은 어떤 단위여도 됩니다. 값이 모두 양수일 때만 로그 스케일도 후보에 들어갑니다.
- 데이터는 **최소 28분기**(훈련 20 + 보정 4 + 정답 4)가 필요합니다.
- 맨 앞이나 맨 뒤의 빈 `Data` 행은 자동으로 잘라 냅니다. **가운데 빈 값**, 숫자가 아닌 값, 모두 같은 값, 순서가 틀리거나 빠지거나 겹치는 분기는 원인과 위치를 알려 주는 오류로 멈춥니다.

**첫 origin의 마지막 분기 날짜**: 이 날짜가 속한 분기까지를 훈련 데이터로 쓰는 것이 첫 origin입니다. 그 뒤로 데이터 끝까지 한 분기씩 origin이 이어집니다.

- 그보다 앞 구간(훈련 최소 20분기부터)은 리샘플링 오차를 쌓는 보정 구간으로만 쓰입니다.
- 조건에 안 맞으면 이유와 가능한 분기를 알려 주는 오류가 납니다.

**첫 origin을 정할 수 있는 범위**

| 제한 | 값 | 이유 |
|---|---|---|
| 가장 이른 분기 | 데이터의 **24번째 분기** (훈련 최소 20분기 + 보정 최소 4분기) | 모델 적합에 훈련 20분기가 필요하고, 리샘플링 오차 풀에 h=4 오차가 하나라도 쌓이려면 첫 origin보다 4개 이전의 origin부터 오차가 필요합니다. |
| 가장 늦은 분기 | 데이터의 **끝에서 5번째 분기** | 첫 origin 뒤에 정답이 4분기 이상 있어야 채점할 수 있습니다. 데이터가 2025Q4까지면 2024Q4입니다. |

- 24분기는 코드가 터지지 않게 하는 **안전 하한**일 뿐 권장값이 아닙니다. 보정 origin이 적으면 리샘플링 오차 풀이 작아서 예측구간이 불안정해집니다. 기존 12시리즈는 보정 15개(첫 origin이 훈련 35분기)로 계산했고, **보정 10개 이상**(첫 origin이 훈련 30분기 이후)을 권합니다.
- 하한은 `PI.py`의 `MIN_CALIB`(보정 최소 개수, 기본 4)와 `selection.py`의 `MIN_TRAIN`(훈련 최소 길이, 기본 20)으로 정해지며, 바꾸려면 이 두 값을 고칩니다. `MIN_TRAIN`을 바꾸면 모델 선정 계산 전체에 영향이 있으므로 `MIN_CALIB`만 바꾸는 것을 권합니다.
- 변수별 가장 이른 분기의 예(데이터 시작이 2011Q1이면 2016Q4, 2011Q2이면 2017Q1, 2010Q2이면 2016Q1).


## 입력 데이터에 문제가 있을 때

입력이 이상하면 계산을 시작하기 전에 원인과 위치를 알려 주는 오류로 멈춥니다. 메시지를 보고 아래처럼 고치세요.

**CSV 파일과 열**

| 상황 | 메시지 | 고치는 법 |
|---|---|---|
| CSV 이름이 틀림 | `CSV를 찾지 못했다: <이름> (현재 폴더, PI_module/data/ 확인)` | 이름 철자와 `data/` 폴더 위치를 확인하세요. 경로를 직접 줘도 됩니다. |
| 열 이름이 틀림 | `CSV에 Date, Data 두 열이 있어야 한다 (없는 열: …, 있는 열: …)` | 열 이름을 `Date`, `Data`로 바꾸세요. 메시지에 지금 있는 열이 나옵니다. |

**값(`Data`)**

| 상황 | 메시지 | 고치는 법 |
|---|---|---|
| 숫자가 아닌 값 | `Data에 숫자가 아닌 값이 있다 (Date 2018-09-01: 'abc')` | 표시된 날짜의 값을 숫자로 고치세요. |
| 가운데에 빈 값 | `Data의 가운데에 빈 값이 있다 (Date 2018-09-01)` | 빈 값을 채우거나 그 구간을 잘라 내세요. 맨 앞과 맨 뒤의 빈 값은 자동으로 잘라 내므로 괜찮습니다. |
| 값이 하나도 없음 | `Data에 값이 없다` | 값이 들어 있는 CSV인지 확인하세요. |
| 값이 모두 같음 | `Data 값이 모두 같아서 모델을 적합할 수 없다` | 변하는 시계열을 넣어야 합니다. |
| 데이터가 짧음 | `데이터가 너무 짧다: 훈련 20 + 보정 4 + 정답 4분기, 최소 28분기가 필요하다 (현재 N분기)` | 28분기 이상의 자료를 쓰세요. |

**날짜(`Date`)**

| 상황 | 메시지 | 고치는 법 |
|---|---|---|
| 날짜로 읽을 수 없음 | `Date를 날짜로 읽지 못했다: …` | `2018-12-01` 같은 형식으로 쓰세요. |
| 분기가 연속이 아님 | `Date가 오름차순으로 연속된 분기가 아니다 (순서, 중복, 빠진 분기를 확인하세요)` | 분기가 빠졌으면 그 분기의 행과 값을 채워 넣고, 겹친 분기는 하나만 남기고, 내림차순이면 오름차순으로 정렬하세요. 한 분기에 한 행이어야 합니다. |

**첫 origin 날짜**

| 상황 | 메시지 | 고치는 법 |
|---|---|---|
| 날짜 형식이 틀림 | `첫 origin 날짜를 읽지 못했다: 'abc' (예: '2019-12-01')` | `2019-12-01`, `2019Q4`, `2019/12/01`, `20191201` 중 하나로 쓰세요. |
| 데이터 범위 밖 | `첫 origin의 분기 2030Q1가 데이터(2011Q1~2025Q4)에 없다` | 데이터 기간 안의 날짜를 쓰세요. |
| 너무 이름 | `첫 origin이 너무 이르다: … 2016Q4 이후여야 한다` | 메시지에 나온 분기 이후로 늦추세요. |
| 너무 늦음 | `첫 origin 뒤에 정답이 4분기 이상 있어야 채점할 수 있다` | 데이터 끝에서 5번째 분기 이전으로 앞당기세요. |


## 출력

**딕셔너리 1** — 예측값

```python
d1["2018-Q4"]       # 2018-Q4까지의 데이터로 만든 예측, h1은 2019-Q1
# 9 x 4 배열: 행 = 5, 10, 20, 30, 50, 70, 80, 90, 95% 분위수 / 열 = h1 ~ h4 (원래 단위)
```

- key는 훈련 데이터의 마지막 분기입니다.
- 첫 origin부터 데이터 끝까지 모든 origin이 들어 있습니다. 마지막 origin은 데이터 끝 이후 4분기를 예측한 값입니다(정답 없음).

**딕셔너리 2** — 통합성적표

```python
{'mse': ..., 'crps': ..., 'coverage': ...}
```

- 4분기 정답이 모두 있는 origin의 평균입니다.
- `mse`, `crps`는 지금까지 노트북과 같이 전체 표준편차로 정규화한 값이고, `coverage`는 90% 구간(5~95%) 적중률(%)입니다.

**저장 파일** (`PI_module/output/<이름>/`, 변수마다 폴더 하나. 같은 변수를 다시 돌리면 같은 이름의 파일을 덮어씁니다) — 파이썬에서는 `json.load`로 읽으면 딕셔너리가 그대로 나옵니다.

| 파일 | 내용 |
|---|---|
| `<이름>_forecast.json` | 딕셔너리 1(예측 분위수) 그대로. `{"2018-Q4": [[h1~h4 5% 분위수], …9줄], …}` |
| `<이름>_score.json` | 딕셔너리 2(통합성적표) 그대로. `{"mse": …, "crps": …, "coverage": …}` |
| `<이름>_forecast.csv` | 모든 origin의 예측 표. 열: `origin, h, 예측 분기, p5 ~ p95` (딕셔너리 1을 그대로 펼친 표) |
| `<이름>_score.csv` | `mse, crps, coverage` |


In [1]:
import json
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
NAME = "X1_l1"                      # PI_module/output/ 안의 예시 결과(X1)
d1 = json.load(open(f"output/{NAME}/{NAME}_forecast.json", encoding="utf-8"))
d2 = json.load(open(f"output/{NAME}/{NAME}_score.json", encoding="utf-8"))
print("딕셔너리 2 (통합성적표):", {k: round(v, 4) for k, v in d2.items()})
key = "2019-Q4"
print(f"\n딕셔너리 1[{key!r}]  -  {key}까지의 데이터로 만든 예측 (행: 분위수, 열: 몇 분기 앞)")
display(pd.DataFrame(d1[key], index=["5%", "10%", "20%", "30%", "50%", "70%", "80%", "90%", "95%"], columns=["h1", "h2", "h3", "h4"]).round(1))
f = pd.read_csv(f"output/{NAME}/{NAME}_forecast.csv")
print("\n_forecast.csv (긴 표) 앞 8행")
display(f.head(8).round(1))
g = f[f["h"] == 1]
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.fill_between(g["예측 분기"], g["p5"], g["p95"], color="#9ec5e8", alpha=.5, label="5~95% 구간")
ax.fill_between(g["예측 분기"], g["p20"], g["p80"], color="#4a90d9", alpha=.35, label="20~80% 구간")
ax.plot(g["예측 분기"], g["p50"], color="#1f4e8c", label="50%")
ax.set_title("h=1 예측 구간 (origin마다 한 분기 앞)", fontweight="bold")
ax.tick_params(axis="x", rotation=60, labelsize=8)
ax.legend(frameon=False, ncol=3)
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
plt.tight_layout(); plt.show()

딕셔너리 2 (통합성적표): {'mse': 1.5448, 'crps': 0.7453, 'coverage': 82.1429}

딕셔너리 1['2019-Q4']  -  2019-Q4까지의 데이터로 만든 예측 (행: 분위수, 열: 몇 분기 앞)


,h1,h2,h3,h4
5%,-151.5,-192.4,-143.0,-150.0
10%,-116.7,-117.9,-134.8,-150.0
20%,17.5,-21.0,-71.2,-79.8
30%,55.1,-0.7,-66.1,-77.4
50%,85.8,49.6,9.6,-5.3
70%,179.9,118.4,112.2,45.6
80%,218.7,211.9,166.5,85.9
90%,297.9,272.2,171.2,147.5
95%,515.0,379.6,243.3,251.6



_forecast.csv (긴 표) 앞 8행


,origin,h,예측 분기,p5,p10,p20,p30,p50,p70,p80,p90,p95
0,2019-Q4,1,2020Q1,-151.5,-116.7,17.5,55.1,85.8,179.9,218.7,297.9,515.0
1,2019-Q4,2,2020Q2,-192.4,-117.9,-21.0,-0.7,49.6,118.4,211.9,272.2,379.6
2,2019-Q4,3,2020Q3,-143.0,-134.8,-71.2,-66.1,9.6,112.2,166.5,171.2,243.3
3,2019-Q4,4,2020Q4,-150.0,-150.0,-79.8,-77.4,-5.3,45.6,85.9,147.5,251.6
4,2020-Q1,1,2020Q2,-263.8,-259.1,-123.5,-73.9,-22.4,104.0,146.5,233.2,470.9
5,2020-Q1,2,2020Q3,-146.2,-142.4,-123.0,-45.4,-10.7,94.2,102.3,248.3,355.9
6,2020-Q1,3,2020Q4,-153.3,-145.2,-121.5,-82.6,-18.2,41.0,151.2,155.8,226.7
7,2020-Q1,4,2021Q1,-152.3,-113.6,-105.1,-83.6,-10.7,37.8,78.6,138.9,240.8


<Figure size 1100x360 with 1 Axes>

## 노트북으로 실행

`PI.ipynb`는 같은 일을 셀 단위로 하는 노트북입니다. 위에서 아래로 실행합니다.

| 셀 | 종류 | 하는 일 | 바꿀 것 |
|---|---|---|---|
| [1] | Input | 분기 데이터 넣기 | `csv_name` |
| [2] | Input | 첫 origin(Out of Sample 시작) 날짜 넣기 | `first_insample_end_date` |
| [3] | Model Selection | 최적 모델 선정 | |
| [4] | Output | 딕셔너리 1: 모든 origin의 예측 분위수 | |
| [5] | Output | 딕셔너리 2: 통합성적표 | |
| [저장] | 선택 | 결과를 JSON, CSV 파일로 저장 | |

각 셀은 두 종류로 나뉩니다.

| 셀 | 표시 | 내용 |
|---|---|---|
| **작동 코드** | 첫 줄 `# ▶ 작동 코드`, 태그 `작동코드` | 계산만 있습니다. 이 셀들만 순서대로 실행하면 `dict1`, `dict2`가 만들어집니다. |
| **설명 코드** | 첫 줄 `# ▷ 설명 코드`, 태그 `설명코드` | 결과를 확인하는 `print`, `display`만 있습니다. 없어도 동작합니다. |

- **노트북만 떼어서 쓰려면** `▶ 작동 코드` 셀만 복사해 가면 됩니다. 같은 폴더에 `selection.py`, `ts_models.py`가 있어야 합니다.
- [저장] 셀은 명령줄(`python PI.py`)과 **같은 폴더(`output/<이름>/`), 같은 이름, 같은 내용**의 파일 4개를 만듭니다(바이트 단위로 같음을 확인했습니다). 명령줄과 노트북 중 어느 쪽으로 실행해도 같은 결과 파일을 공유하고, 같은 입력이면 나중에 실행한 쪽이 덮어씁니다.


## 파이썬에서 쓰기

```python
from PI import run_pi, to_frame

d1, d2 = run_pi("gold_yoy", "2018-12-01")     # CSV 이름 또는 DataFrame(Date, Data)
to_frame(d1, "gold_yoy")                       # d1을 긴 표로 (두 번째 인자로 CSV 이름을 주면 정답 actual 열이 추가됨)
```


## 테스트

`test_PI.py`가 오류 처리와 결과 형태를 검사합니다. PI_module 폴더에서 실행합니다.

```
python test_PI.py          # 빠른 테스트 (28개, 캐시를 쓰므로 금방 끝남)
python test_PI.py full     # 위 + 변수별 가장 이른/늦은/중간 날짜 8개 (새로 계산해서 30분 안팎, 총 36개)
```

- 빠른 테스트: 오류가 나야 하는 경우(날짜 범위·형식, 열 없음, 분기 빠짐·중복·정렬, 빈 값, 문자 값, 상수, 28분기 미만, 명령줄 인자), 입력 표기 변형(`2019Q4`, `2019/12/01`, 경로 형식 등)이 같은 결과를 내는지.
- `full`: 빠른 테스트 28개에 **변수별 날짜 8개**를 더해 모델 선정부터 새로 계산합니다. 첫 origin 날짜가 바뀌면 보정 구간 길이가 달라져 캐시를 못 쓰기 때문에 30분 안팎 걸립니다.

  | 변수 | 날짜 |
  |---|---|
  | `X2`, `X6_l1` | 허용되는 가장 이른 날짜, 가장 늦은 날짜 (2개씩) |
  | `X1_l1`, `X3_l1`, `X4`, `X5_l1` | 중간 날짜 (훈련 30분기, 보정 10개) |

  날짜마다 딕셔너리 1의 key 개수, 9×4 배열, 분위수 순서, 정답 없는 예측 10개, 딕셔너리 1로 다시 센 커버리지가 딕셔너리 2와 같은지를 확인합니다. 가장 늦은 날짜는 채점 origin이 1개뿐이라 커버리지 수치에 의미는 없고, 오류 없이 도는지만 봅니다.
- **평소에는 빠른 테스트만** 돌리면 됩니다. `full`은 모델 계산 코드를 크게 고쳤을 때만 돌리세요.
- 마지막 줄에 `총 N개 중 통과 N개, 실패 0개`가 나오면 정상이고, 실패가 있으면 그 아래에 항목과 이유가 나옵니다. VS Code에서 파일을 열고 ▶ 버튼으로 실행해도 빠른 테스트가 돌아갑니다.
- 코드를 고친 뒤에 다시 돌려 보는 용도이며, PI 실행에는 필요 없습니다.


## 작동 과정

`python PI.py gold_yoy '2024-01-01'`을 실행하면 아래 순서로 진행됩니다.

```
CSV ─▶ ① 입력 읽기 ─▶ ② 첫 origin 정하기 ─▶ ③ 모든 조합 백테스트 ─▶ ④ 모델 선정
                                                   │ (캐시)                  │
                                                   ▼                         ▼
 PI_module/output/ ◀─ ⑦ 저장·출력 ◀─ ⑥ 딕셔너리 1·2 만들기 ◀─ ⑤ 선정 조합만 다시 계산(분위수)
```

| 단계 | 하는 일 | 어디서 |
|---|---|---|
| ① 입력 읽기 | CSV의 `Date`를 분기로 바꾸고, 분기가 연속인지 확인합니다. | `PI.py`의 `read_series` |
| ② 첫 origin 정하기 | 입력 날짜의 분기 위치로 첫 origin(훈련 관측 수)을 정하고, 그 앞 구간을 보정 origin 수(첫 origin − 20)로 셉니다. 범위를 벗어나면 오류를 냅니다. | `PI.py`의 `run_pi` |
| ③ 모든 조합 백테스트 | 모델 3종 × (d, D) 4가지 × 스케일(원, 로그) 조합마다 origin을 한 분기씩 늘려 가며 적합하고, h=1~4를 예측합니다. 보정 구간에서는 예측 오차만 쌓고, 첫 origin부터는 CRPS·MSE·커버리지를 채점합니다. 구간 방식(자체, 리샘플링)은 같은 적합에서 둘 다 계산합니다. | `selection.py`의 `run_grid`, 모델은 `ts_models.py` |
| ④ 모델 선정 | 조합별 평균 점수를 표로 만들어, CRPS 1등의 ±10% 이내 중 커버리지가 90%에 가장 가까운 조합을 고릅니다. | `selection.py`의 `select` |
| ⑤ 선정 조합만 다시 계산 | 고른 조합 하나를 같은 시드(42)로 다시 돌려 origin마다 5~95%의 9개 분위수를 뽑습니다. 자체 구간은 `예측값 ± z × σ`, 리샘플링 구간은 샘플의 분위수입니다. | `selection.py`의 `_run_one(levels=…)` |
| ⑥ 딕셔너리 만들기 | 딕셔너리 1은 origin의 마지막 분기를 key로 9×4 배열을 담고, 딕셔너리 2는 선정 조합의 평균 mse·crps·coverage입니다. | `PI.py`의 `run_pi` |
| ⑦ 저장·출력 | 화면에 출력하고 `PI_module/output/<이름>/`에 JSON과 CSV로 저장합니다. | `PI.py` |

**각 origin에서 일어나는 일 (③과 ⑤의 안쪽)**

1. 그 시점까지의 데이터(훈련 구간)만 쓰고, 훈련 구간 표준편차로 단위를 맞춥니다(미래 정보 없음).
2. 모델을 적합해 h=1~4의 예측값과 표준편차(σ)를 얻습니다.
3. 보정 구간이면 정답과의 오차를 오차 풀에 쌓기만 합니다.
4. 첫 origin부터는 오차 풀에서 500개를 뽑아 예측분포를 만들고, 자체 구간과 리샘플링 구간을 각각 계산해 정답과 비교합니다.
5. 정답이 아직 없는 마지막 origin들은 예측만 하고 채점하지 않습니다(딕셔너리 1에는 들어갑니다).

**캐시**

- ③의 결과는 `PI_module/cache/`에 저장됩니다. 같은 CSV와 같은 첫 origin으로 다시 실행하면 계산 없이 바로 읽습니다.
- CSV에 새 분기를 덧붙이거나 값이 바뀌면, 바뀐 시점의 영향을 받는 origin부터 뒤쪽만 다시 계산합니다(결과는 처음부터 계산한 것과 같습니다).
- 첫 origin 날짜를 바꾸면 보정 구간 길이가 달라지므로 새로 계산합니다.


In [2]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
COL = {"PI.py": "#cfe3f7", "selection.py": "#d5ecd4", "둘 다": "#fde3c1"}
steps = [("① 입력 읽기", "CSV → 분기, 값", "PI.py"), ("② 첫 origin 정하기", "날짜 → 훈련 길이, 보정 구간", "PI.py"),
         ("③ 모든 조합 백테스트", "모델 3 × (d, D) 4 × 스케일\n→ 캐시에 저장", "둘 다"), ("④ 모델 선정", "CRPS ±10% 중\n커버리지 90%에 가까운 것", "selection.py"),
         ("⑤ 선정 조합 다시 계산", "시드 42, 9개 분위수", "selection.py"), ("⑥ 딕셔너리 만들기", "딕셔너리 1 (9×4 분위수)\n딕셔너리 2 (mse, crps, coverage)", "PI.py"),
         ("⑦ 저장·출력", "화면, JSON, CSV → output/<이름>/", "PI.py")]
pos = [(0, 1), (1, 1), (2, 1), (3, 1), (3, 0), (2, 0), (1, 0)]
fig, ax = plt.subplots(figsize=(13, 4.6)); ax.set_xlim(-0.55, 3.55); ax.set_ylim(-0.6, 1.6); ax.axis("off")
for (t, d, f), (x, y) in zip(steps, pos):
    ax.add_patch(FancyBboxPatch((x - 0.4, y - 0.32), 0.8, 0.64, boxstyle="round,pad=0.02,rounding_size=0.06", fc=COL[f], ec="#555", lw=1.2))
    ax.text(x, y + 0.14, t, ha="center", va="center", fontsize=11.5, fontweight="bold", color="#172033")
    ax.text(x, y - 0.1, d, ha="center", va="center", fontsize=8.8, color="#333")
    ax.text(x, y - 0.27, f, ha="center", va="center", fontsize=8, color="#666", style="italic")
for (x0, y0), (x1, y1) in zip(pos[:-1], pos[1:]):
    if y0 == y1: a, b = (x0 + (0.42 if x1 > x0 else -0.42), y0), (x1 - (0.42 if x1 > x0 else -0.42), y1)
    else: a, b = (x0, y0 - 0.34), (x1, y1 + 0.34)
    ax.annotate("", xy=b, xytext=a, arrowprops=dict(arrowstyle="-|>", color="#555", lw=1.4))
ax.text(-0.5, 1.55, "CSV (Date, Data) + 첫 origin 날짜", fontsize=10, color="#172033", fontweight="bold", va="top")
ax.text(0.0, -0.55, "파란색 = PI.py   초록색 = selection.py(모델은 ts_models.py)   주황색 = 둘 다", fontsize=9, color="#555")
plt.show()

<Figure size 1300x460 with 1 Axes>

## 모델 선정 방식 (지금까지와 같음)

- 후보: 모델 3종(AR 단독, GARCH(1,1), SV) × 차분 (d, D) 4가지 × 스케일(원, 로그) × 구간 방식(모델 자체, 리샘플링)
- 평가: 첫 origin부터 확장 윈도우 백테스트, h=1~4, CRPS·MSE·커버리지
- 선정: CRPS 1등의 ±10% 이내 후보 중 커버리지가 90%에 가장 가까운 조합
- 리샘플링 난수 시드는 42로 고정되어 같은 입력이면 같은 결과가 나옵니다.
- 첫 실행은 조합을 모두 계산해서 몇 분 걸리고(길이 60분기 안팎에서 약 2분), 같은 입력으로 다시 실행하면 캐시(`PI_module/cache/`)를 읽어 바로 끝납니다.


## 알아 둘 점

- 모델 선정과 성적 평가가 같은 구간(첫 origin 이후)이라 점수가 다소 낙관적일 수 있습니다.
- horizon은 h=1~4로 고정입니다.
- 필요한 패키지: numpy, pandas, scipy, statsmodels, arch, pmdarima, properscoring (프로젝트 환경과 같음)


## 폴더

```
PI_module/
├─ PI.py          실행 파일(명령줄과 함수 겸용)
├─ PI.ipynb       셀 단위 노트북([1] 데이터 → [2] 첫 origin 날짜 → [3] 모델 선정 → [4] 딕셔너리 1 → [5] 딕셔너리 2 → [저장])
├─ selection.py   모델 선정 엔진 (프로젝트의 model_selection/selection.py 복사본)
├─ ts_models.py   AR 단독, GARCH, SV 모델 구현 (model_selection/ts_models.py 복사본)
├─ test_PI.py     오류·결과 검사 (python test_PI.py)
├─ README.md / README.ipynb   설명서(같은 내용, 노트북은 보기 편하게)
├─ data/          입력 CSV
├─ cache/         계산 캐시 (실행하면 생김)
└─ output/        결과 (변수별 폴더, JSON, CSV)
```

- 이 폴더만 통째로 옮겨도 실행됩니다. 프로젝트의 다른 폴더는 필요 없습니다.
- `selection.py`와 `ts_models.py`는 프로젝트의 `model_selection/`에서 **복사한 것**이라, 한쪽을 고쳐도 다른 쪽에는 자동으로 반영되지 않습니다. 복사본에서 바뀐 부분은 `selection.py`의 import 한 줄(`from ts_models import ...`)과 캐시 위치(`PI_module/cache/`)뿐입니다.
